In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

In [2]:
#=========
# Setup
#=========
df = pd.DataFrame({
    "order_id": [101 , 102 , 102 , 103 , 104 , 105] ,
    "customer_id": [1 , 2 , 2 , 3 , 4 , 4] ,
    "region": ["East" , "West" , "West" , "East" , None , "West"] ,
    "order_date": pd.to_datetime(["2025-12-01" , "2025-12-02" , "2025-12-02" , "2025-12-04" , "2025-12-06" , None]) ,
    "sales": [120 , 250 , 250 , np.nan , 90 , 110] ,
    "returns": [5 , np.nan , np.nan , 2 , 3 , np.nan] ,
})
df

,order_id,customer_id,region,order_date,sales,returns
0,101,1,East,2025-12-01,120.0,5.0
1,102,2,West,2025-12-02,250.0,NaN
2,102,2,West,2025-12-02,250.0,NaN
3,103,3,East,2025-12-04,NaN,2.0
4,104,4,None,2025-12-06,90.0,3.0
5,105,4,West,NaT,110.0,NaN


In [3]:
#============================================================
# Case 1) Detect missing values + quick completeness scan
#============================================================
missing_count = df.isna().sum().sort_values(ascending = False)
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending = False)
case1 = pd.DataFrame({"missing_count": missing_count , "missing_pct": missing_pct})
case1

,missing_count,missing_pct
returns,3,50.0
region,1,16.7
order_date,1,16.7
sales,1,16.7
order_id,0,0.0
customer_id,0,0.0


In [5]:
#======================================
# Case 2) dropna() with subset/thresh
#======================================
case2a = df.dropna(subset = ["order_id" , "order_date"])
case2a

case2b = df.dropna(thresh = 5)
case2b

,order_id,customer_id,region,order_date,sales,returns
0,101,1,East,2025-12-01,120.0,5.0
1,102,2,West,2025-12-02,250.0,NaN
2,102,2,West,2025-12-02,250.0,NaN
3,103,3,East,2025-12-04,NaN,2.0
4,104,4,None,2025-12-06,90.0,3.0


,order_id,customer_id,region,order_date,sales,returns
0,101,1,East,2025-12-01,120.0,5.0
1,102,2,West,2025-12-02,250.0,NaN
2,102,2,West,2025-12-02,250.0,NaN
3,103,3,East,2025-12-04,NaN,2.0
4,104,4,None,2025-12-06,90.0,3.0


In [8]:
#==========================================================
# Case 3) fillna(): scalar + dict + ffill/bfill patterns
#==========================================================
case3 = df.copy()
case3["returns"] = case3["returns"].fillna(0)
case3["region"] = case3["region"].fillna("Unknown")
case3["sales"].median()
case3["sales"] = case3["sales"].fillna(case3["sales"].median())
case3

np.float64(120.0)

,order_id,customer_id,region,order_date,sales,returns
0,101,1,East,2025-12-01,120.0,5.0
1,102,2,West,2025-12-02,250.0,0.0
2,102,2,West,2025-12-02,250.0,0.0
3,103,3,East,2025-12-04,120.0,2.0
4,104,4,Unknown,2025-12-06,90.0,3.0
5,105,4,West,NaT,110.0,0.0


In [10]:
#========================
# Case 4) interpolate()
#========================
ts = (
    df[["order_date" , "sales"]].dropna(subset = ["order_date"])
        .sort_values("order_date").set_index("order_date")
)

case4 = ts.copy()
case4["sales_interp"] = case4["sales"].interpolate(limit = 1)
case4

,sales,sales_interp
order_date,,
2025-12-01,120.0,120.0
2025-12-02,250.0,250.0
2025-12-02,250.0,250.0
2025-12-04,NaN,170.0
2025-12-06,90.0,90.0


In [12]:
#============================
# Case 5) Deduplicate rows
#============================
dupe_mask = df.duplicated(subset = ["order_id"] , keep = "last")
case5_dupes = df.loc[dupe_mask]
case5_dupes

case5 = df.drop_duplicates(subset = ["order_id"] , keep = "last")
case5

,order_id,customer_id,region,order_date,sales,returns
1,102,2,West,2025-12-02,250.0,NaN


,order_id,customer_id,region,order_date,sales,returns
0,101,1,East,2025-12-01,120.0,5.0
2,102,2,West,2025-12-02,250.0,NaN
3,103,3,East,2025-12-04,NaN,2.0
4,104,4,None,2025-12-06,90.0,3.0
5,105,4,West,NaT,110.0,NaN


In [13]:
#============================
# Case 6) Duplicate labels
#============================
dfl = df.set_index("order_id")
dfl.index.is_unique

case6 = dfl.loc[~dfl.index.duplicated(keep = "first")]
case6

False

,customer_id,region,order_date,sales,returns
order_id,,,,,
101,1,East,2025-12-01,120.0,5.0
102,2,West,2025-12-02,250.0,NaN
103,3,East,2025-12-04,NaN,2.0
104,4,None,2025-12-06,90.0,3.0
105,4,West,NaT,110.0,NaN
